In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TEMP VIEW customers_typed AS
SELECT
  customer_id as customer_id,
  customer_unique_id as customer_unique_id,
  customer_zip_code_prefix as customer_zip_code_prefix,
  customer_state,
  customer_city,
  _ingested_at
FROM workspace.bronze.customers
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE workspace.silver.customers AS
SELECT * FROM customers_typed
WHERE customer_id IS NOT NULL AND customer_unique_id IS NOT NULL;

CREATE OR REPLACE TABLE workspace.silver.customers_quarantine AS
SELECT *, 'Llave de customer o identificador de customer nulo' AS motivo
FROM customers_typed
WHERE customer_id IS NULL OR customer_unique_id IS NULL;